In [0]:
# === J7 - Databricks Delta Lake pipeline setup ===
import sys
import os

repo_root = "/Workspace/Users/salmacharki721@gmail.com/cybersecurity-data-pipeline"
sys.path.insert(0, repo_root)

import importlib
for mod in ['src.ingestion', 'src.cleaning', 'src.transformation']:
    if mod in sys.modules:
        importlib.reload(sys.modules[mod])

from src.ingestion import ingest_parquet_to_bronze_memory
from src.cleaning import clean_bronze_to_silver
from src.transformation import transform_silver_to_gold

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("j7-delta-pipeline").getOrCreate()
spark.sql("USE CATALOG workspace")

print(f"✓ Spark version: {spark.version}")
print(f"✓ Catalog: workspace")
print(f"✓ Modules imported")

In [0]:
# === Verify Unity Catalog setup ===
print("--- Available schemas in workspace catalog ---")
spark.sql("SHOW SCHEMAS").show()

print("--- Schemas in workspace ---")
spark.sql("SHOW SCHEMAS IN workspace").show()

print("--- Tables in workspace.cybersecurity ---")
try:
    spark.sql("SHOW TABLES IN workspace.cybersecurity").show()
except Exception as e:
    print(f"  Schema doesn't exist yet, will create: {e}")

print("--- Volumes in workspace.cybersecurity ---")
spark.sql("SHOW VOLUMES IN workspace.cybersecurity").show()

# Create schema if not exists
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.cybersecurity")
print("✓ Schema workspace.cybersecurity ready")

In [0]:
# === Read DistriNet Monday from Unity Catalog Volume ===
disternet_path = "/Volumes/workspace/cybersecurity/raw_data/Benign-Monday.parquet"

print(f"Reading: {disternet_path}")

# Bronze ingestion (in memory)
df_bronze = ingest_parquet_to_bronze_memory(spark, disternet_path)

print(f"\n✓ Bronze DataFrame (in memory)")
print(f"  Rows: {df_bronze.count()}")
print(f"  Columns: {len(df_bronze.columns)}")

In [0]:
# === Write Bronze as Delta table in Unity Catalog ===
print("Writing Bronze table as Delta...")

(df_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.cybersecurity.bronze"))

print("✓ Bronze Delta table created: workspace.cybersecurity.bronze")

# Verify
print("\n--- Table info ---")
spark.sql("DESCRIBE TABLE EXTENDED workspace.cybersecurity.bronze").show(truncate=False)

print("\n--- Row count ---")
spark.sql("SELECT COUNT(*) AS total_rows FROM workspace.cybersecurity.bronze").show()

print("\n--- Schema preview (first 10 columns) ---")
bronze_cols = spark.sql("SELECT * FROM workspace.cybersecurity.bronze LIMIT 5")
bronze_cols.select(bronze_cols.columns[:10]).show()

In [0]:
# === Read Bronze from Delta table and run cleaning ===
print("Reading Bronze from Delta table...")

df_bronze = spark.read.table("workspace.cybersecurity.bronze")

print(f"✓ Bronze loaded from Delta: {df_bronze.count()} rows")

# Run Bronze -> Silver cleaning
print("\nRunning Silver cleaning pipeline...")
print("=" * 60)
df_silver = clean_bronze_to_silver(df_bronze)
print("=" * 60)

print(f"\n✓ Silver DataFrame ready: {df_silver.count()} rows, {len(df_silver.columns)} columns")

In [0]:
# === Write Silver as Delta table ===
print("Writing Silver table as Delta...")

(df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.cybersecurity.silver"))

print("✓ Silver Delta table created: workspace.cybersecurity.silver")

# Verify
print("\n--- Tables in workspace.cybersecurity ---")
spark.sql("SHOW TABLES IN workspace.cybersecurity").show()

print("--- Silver row count ---")
spark.sql("SELECT COUNT(*) AS total_rows FROM workspace.cybersecurity.silver").show()

print("--- Label distribution (Silver) ---")
spark.sql("""
    SELECT label, COUNT(*) AS count
    FROM workspace.cybersecurity.silver
    GROUP BY label
    ORDER BY count DESC
""").show()

In [0]:
# === Read Silver from Delta table and run feature engineering ===
print("Reading Silver from Delta table...")

df_silver = spark.read.table("workspace.cybersecurity.silver")

print(f"✓ Silver loaded from Delta: {df_silver.count()} rows")

# Run Silver -> Gold feature engineering
print("\nRunning Gold transformation pipeline...")
print("=" * 60)
df_gold = transform_silver_to_gold(df_silver)
print("=" * 60)

print(f"\n✓ Gold DataFrame ready: {df_gold.count()} rows, {len(df_gold.columns)} columns")

In [0]:
# === Write Gold as Delta table ===
print("Writing Gold table as Delta...")

(df_gold.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.cybersecurity.gold"))

print("✓ Gold Delta table created: workspace.cybersecurity.gold")

# Verify
print("\n--- All tables in workspace.cybersecurity ---")
spark.sql("SHOW TABLES IN workspace.cybersecurity").show()

print("--- Gold row count ---")
spark.sql("SELECT COUNT(*) AS total_rows FROM workspace.cybersecurity.gold").show()

print("--- Gold column count ---")
gold_cols = spark.sql("SELECT * FROM workspace.cybersecurity.gold LIMIT 1").columns
print(f"  Total columns: {len(gold_cols)}")
print(f"  First 10: {gold_cols[:10]}")
print(f"  New features (Gold-only): {[c for c in gold_cols if c not in df_silver.columns]}")

In [0]:
# === Query the Delta tables with SQL ===
print("=" * 60)
print("SQL Queries on Delta tables")
print("=" * 60)

print("\n--- 1. Bronze: total flows ---")
spark.sql("""
    SELECT COUNT(*) AS bronze_rows,
           COUNT(DISTINCT _source_file) AS source_files
    FROM workspace.cybersecurity.bronze
""").show()

print("\n--- 2. Silver: label distribution ---")
spark.sql("""
    SELECT label, COUNT(*) AS count
    FROM workspace.cybersecurity.silver
    GROUP BY label
    ORDER BY count DESC
""").show()

print("\n--- 3. Gold: is_attack distribution ---")
spark.sql("""
    SELECT is_attack, COUNT(*) AS count
    FROM workspace.cybersecurity.gold
    GROUP BY is_attack
    ORDER BY is_attack
""").show()

print("\n--- 4. Gold: protocol distribution ---")
spark.sql("""
    SELECT protocol_name, COUNT(*) AS count,
           ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM workspace.cybersecurity.gold), 2) AS percentage
    FROM workspace.cybersecurity.gold
    GROUP BY protocol_name
    ORDER BY count DESC
""").show()

print("\n--- 5. Gold: top 5 protocols by total bytes ---")
spark.sql("""
    SELECT protocol_name,
           COUNT(*) AS flow_count,
           SUM(total_bytes) AS total_bytes_sum,
           SUM(total_packets) AS total_packets_sum,
           ROUND(AVG(bytes_per_packet), 2) AS avg_bytes_per_packet
    FROM workspace.cybersecurity.gold
    GROUP BY protocol_name
    ORDER BY total_bytes_sum DESC
    LIMIT 5
""").show()

print("\n--- 6. Gold: average bytes per protocol ---")
spark.sql("""
    SELECT protocol_name,
           COUNT(*) AS flow_count,
           ROUND(AVG(total_bytes), 2) AS avg_bytes,
           ROUND(AVG(total_packets), 2) AS avg_packets
    FROM workspace.cybersecurity.gold
    GROUP BY protocol_name
    ORDER BY flow_count DESC
""").show()

In [0]:
# === Delta Lake advanced features demo ===
print("=" * 60)
print("Delta Lake Features Demo")
print("=" * 60)

print("\n--- 1. Table history (DESCRIBE HISTORY) ---")
spark.sql("DESCRIBE HISTORY workspace.cybersecurity.gold").show(truncate=False)

print("\n--- 2. Table details (location, format) ---")
spark.sql("DESCRIBE DETAIL workspace.cybersecurity.gold").show(truncate=False, vertical=True)

print("\n--- 3. Time travel: read version 0 of Gold table ---")
try:
    v0_count = spark.sql("SELECT COUNT(*) FROM workspace.cybersecurity.gold VERSION AS OF 0").collect()[0][0]
    print(f"  Version 0 row count: {v0_count}")
except Exception as e:
    print(f"  Time travel demo: {e}")

print("\n--- 4. Optimize the Gold table (compact small files) ---")
try:
    spark.sql("OPTIMIZE workspace.cybersecurity.gold")
    print("  ✓ Gold table optimized")
except Exception as e:
    print(f"  Optimize (may not be supported on Serverless): {e}")

print("\n--- 5. Final table summary ---")
spark.sql("""
    SELECT 
        'bronze' AS layer,
        COUNT(*) AS row_count
    FROM workspace.cybersecurity.bronze
    UNION ALL
    SELECT 
        'silver' AS layer,
        COUNT(*) AS row_count
    FROM workspace.cybersecurity.silver
    UNION ALL
    SELECT 
        'gold' AS layer,
        COUNT(*) AS row_count
    FROM workspace.cybersecurity.gold
""").show()